In [1]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/capstone-ml'
DATA = f'{ROOT}/data'
PRED = f'{ROOT}/predictions'
LOGS = f'{ROOT}/logs'

import sys, importlib.metadata as md
print('Python', sys.version.split()[0])
for p in ['transformers', 'datasets', 'scikit-learn', 'pandas', 'numpy']:
    print(f'{p}=={md.version(p)}')

Mounted at /content/drive
Python 3.13.18
transformers==5.22.1
datasets==4.8.5
scikit-learn==1.6.1
pandas==2.2.3
numpy==2.4.6


In [2]:
import pandas as pd
from datasets import load_dataset

ds = load_dataset('SetFit/sst5')

COLLAPSE = {0: 0, 1: 0, 2: 1, 3: 2, 4: 2}   # very neg, neg -> neg; neutral; pos, very pos -> pos
NAMES = {0: 'negative', 1: 'neutral', 2: 'positive'}

raw = {}
for split, hf_name in [('train', 'train'), ('val', 'validation'), ('test', 'test')]:
    df = ds[hf_name].to_pandas()[['text', 'label']].rename(columns={'label': 'label5'})
    df.insert(0, 'id', [f'sst5-{split}-{i:05d}' for i in range(len(df))])
    df['label'] = df['label5'].map(COLLAPSE)
    raw[split] = df

summary = pd.DataFrame({s: raw[s]['label'].map(NAMES).value_counts() for s in raw}).loc[['negative', 'neutral', 'positive']]
summary.loc['total'] = summary.sum()
print(summary)
print()
print('Five-class train counts:', raw['train']['label5'].value_counts().sort_index().to_dict())

README.md:   0%|          | 0.00/421 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/1.32M [00:00<?, ?B/s]

dev.jsonl:   0%|          | 0.00/171k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/343k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8544 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1101 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2210 [00:00<?, ? examples/s]

          train   val  test
label                      
negative   3310   428   912
neutral    1624   229   389
positive   3610   444   909
total      8544  1101  2210

Five-class train counts: {0: 1092, 1: 2218, 2: 1624, 3: 2322, 4: 1288}


In [3]:
assert len(raw['train']) == 8544 and len(raw['val']) == 1101 and len(raw['test']) == 2210
train_counts = raw['train']['label'].value_counts().sort_index().tolist()
assert train_counts == [3310, 1624, 3610], train_counts
print('Train counts confirmed: 3,310 negative, 1,624 neutral, 3,610 positive')
print(f"Neutral share of training data: {train_counts[1] / sum(train_counts):.1%}")

Train counts confirmed: 3,310 negative, 1,624 neutral, 3,610 positive
Neutral share of training data: 19.0%


In [4]:
import re

train_text = raw['train']['text']

patterns = {
    "split negation, e.g. does n't":      r" n't\b",
    "split clitic, e.g. it 's":            r" '(s|re|ve|ll|d|m)\b",
    "space before punctuation, e.g. x ,":  r" [.,!?;:]",
    "bracket tokens, e.g. -lrb-":          r"-[lr][rsc]b-",
    "HTML entities, e.g. &amp;":           r"&[a-z]+;",
    "any uppercase letter":                r"[A-Z]",
    "any non-ASCII character":             r"[^\x00-\x7f]",
}
for name, pat in patterns.items():
    n = train_text.str.contains(pat, regex=True).sum()
    print(f'{n:5d} of 8544  {name}')

print('\nFive example sentences:')
for t in train_text.sample(5, random_state=1):
    print(' ', t)

  647 of 8544  split negation, e.g. does n't
 2449 of 8544  split clitic, e.g. it 's
 8454 of 8544  space before punctuation, e.g. x ,
  344 of 8544  bracket tokens, e.g. -lrb-
    0 of 8544  HTML entities, e.g. &amp;
    0 of 8544  any uppercase letter
   90 of 8544  any non-ASCII character

Five example sentences:
  this overproduced piece of dreck is shockingly bad and absolutely unnecessary .
  beautifully crafted and cooly unsettling ... recreates the atmosphere of the crime expertly .
  a sincere but dramatically conflicted gay coming-of-age tale .
  like a precious and finely cut diamond , magnificent to behold in its sparkling beauty yet in reality it 's one tough rock .
  bearable .


/tmp/ipykernel_399/1051486136.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  n = train_text.str.contains(pat, regex=True).sum()


In [5]:
from collections import Counter

chars = Counter(c for t in train_text for c in t if ord(c) > 127)
print('Non-ASCII characters and how often they appear:')
for c, n in chars.most_common():
    print(f'  {c!r}  U+{ord(c):04X}  {n}')

print('\nExamples:')
for t in train_text[train_text.str.contains(r'[^\x00-\x7f]', regex=True)].head(5):
    print(' ', t)

print('\nTreebank quote marks:', train_text.str.contains('``').sum(), 'with ``,',
      train_text.str.contains("''").sum(), "with ''")

Non-ASCII characters and how often they appear:
  'é'  U+00E9  69
  'â'  U+00E2  4
  'ï'  U+00EF  4
  '\xa0'  U+00A0  3
  'è'  U+00E8  3
  'á'  U+00E1  3
  'ñ'  U+00F1  3
  'ó'  U+00F3  2
  'ü'  U+00FC  2
  'í'  U+00ED  1
  'æ'  U+00E6  1
  'ã'  U+00E3  1
  'ç'  U+00E7  1
  'ô'  U+00F4  1

Examples:
  béart and berling are both superb , while huppert ... is magnificent .
  the most repugnant adaptation of a classic text since roland joffé and demi moore 's the scarlet letter .
  the film is like sitting in a downtown café , overhearing a bunch of typical late-twenty-somethings natter on about nothing , and desperately wishing you could change tables .
  contrived pastiche of caper clichés .
  garcía bernal and talancón are an immensely appealing couple , and even though their story is predictable , you 'll want things to work out .

Treebank quote marks: 223 with ``, 223 with ''


In [10]:
import html, unicodedata

PTB_BRACKETS = {'-lrb-': '(', '-rrb-': ')', '-lsb-': '[', '-rsb-': ']', '-lcb-': '{', '-rcb-': '}'}
CHAR_MAP = {'\u2018': "'", '\u2019': "'", '\u201c': '"', '\u201d': '"',
            '\u2013': '-', '\u2014': '-', '\u00a0': ' '}

def clean_text(text: str) -> str:
    t = html.unescape(str(text))                                # &amp; -> &  (marketplace only)
    t = re.sub(r'<[^>]+>', ' ', t)                              # HTML tags   (marketplace only)
    t = unicodedata.normalize('NFKC', t)                        # one form per character, accents kept
    t = ''.join(CHAR_MAP.get(c, c) for c in t)                  # curly quotes and long dashes to plain
    t = re.sub(r'[\u0000-\u0008\u000b-\u001f\u007f\u200b-\u200f\ufeff]', '', t)  # invisible characters
    t = t.lower()                                               # SST-5 has 0 uppercase letters
    for tok, ch in PTB_BRACKETS.items():
        t = t.replace(tok, ch)                                  # 344 sentences
    t = re.sub(r'``\s*', '"', t)                                # 223 sentences: `` the ring '' -> "the ring"
    t = re.sub(r"\s*''", '"', t)
    t = re.sub(r" n't\b", "n't", t)                             # 647 sentences
    t = re.sub(r" '(s|re|ve|ll|d|m)\b", r"'\1", t)              # 2,449 sentences
    t = re.sub(r' ([.,!?;:%)\]}])', r'\1', t)                   # 8,454 sentences
    t = re.sub(r'([(\[{$]) ', r'\1', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

tests = {
    "it 's a movie that does n't quite work , but -lrb- mostly -rrb- fun .": "it's a movie that doesn't quite work, but (mostly) fun.",
    "ca n't wait ; wo n't stop !": "can't wait; won't stop!",
    "Fixed the leak &amp; left it <b>clean</b>.  Will call him again": "fixed the leak & left it clean. will call him again",
    "He\u2019s a good fundi \u2014 paid via M-Pesa": "he's a good fundi - paid via m-pesa",
    "a remake of `` the ring '' , sadly .": 'a remake of "the ring", sadly.',
}
for src, want in tests.items():
    got = clean_text(src)
    assert got == want, (src, got)
print('clean_text passes all', len(tests), 'checks')

clean_text passes all 5 checks


In [11]:
clean = {}
for split, df in raw.items():
    out = df.copy()
    out['text'] = out['text'].map(clean_text)
    out['source'] = 'sst5'
    clean[split] = out
    changed = (out['text'] != df['text']).sum()
    empty = (out['text'].str.len() == 0).sum()
    print(f'{split}: {changed} of {len(df)} texts changed, {empty} empty after cleaning')

assert all((c['text'].str.len() > 0).all() for c in clean.values())

print('\nBefore and after, five training examples:')
diff = raw['train']['text'] != clean['train']['text']
for b, a in zip(raw['train'].loc[diff, 'text'].head(5), clean['train'].loc[diff, 'text'].head(5)):
    print('  before:', b)
    print('  after: ', a, '\n')

train: 8482 of 8544 texts changed, 0 empty after cleaning
val: 1092 of 1101 texts changed, 0 empty after cleaning
test: 2192 of 2210 texts changed, 0 empty after cleaning

Before and after, five training examples:
  before: a stirring , funny and finally transporting re-imagining of beauty and the beast and 1930s horror films
  after:  a stirring, funny and finally transporting re-imagining of beauty and the beast and 1930s horror films 

  before: apparently reassembled from the cutting-room floor of any given daytime soap .
  after:  apparently reassembled from the cutting-room floor of any given daytime soap. 

  before: they presume their audience wo n't sit still for a sociology lesson , however entertainingly presented , so they trot out the conventional science-fiction elements of bug-eyed monsters and futuristic women in skimpy clothes .
  after:  they presume their audience won't sit still for a sociology lesson, however entertainingly presented, so they trot out the conventio

In [9]:
for split, df in clean.items():
    dup = df['text'].duplicated(keep=False)
    conflicting = df[dup].groupby('text')['label'].nunique().gt(1).sum()
    print(f'{split}: {dup.sum()} rows share their text with another row; {conflicting} texts carry conflicting labels')

sets = {s: set(df['text']) for s, df in clean.items()}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    print(f'{a} and {b} share {len(sets[a] & sets[b])} identical texts')

train: 20 rows share their text with another row; 0 texts carry conflicting labels
val: 2 rows share their text with another row; 0 texts carry conflicting labels
test: 0 rows share their text with another row; 0 texts carry conflicting labels
train and val share 1 identical texts
train and test share 2 identical texts
val and test share 0 identical texts


In [12]:
from transformers import AutoTokenizer

words = clean['train']['text'].str.split().str.len()
print('Words per training sentence: median', int(words.median()),
      '| 95th percentile', int(words.quantile(0.95)), '| longest', int(words.max()))
print(f'Over 64 words (the LSTM cuts these): {(words > 64).mean():.2%}')

for name in ['distilbert-base-uncased', 'roberta-base']:
    tok = AutoTokenizer.from_pretrained(name)
    n = [len(x) for x in tok(clean['train']['text'].tolist())['input_ids']]
    print(f'{name}: longest {max(n)} tokens, {sum(i > 128 for i in n) / len(n):.2%} over 128')

Words per training sentence: median 16 | 95th percentile 32 | longest 49
Over 64 words (the LSTM cuts these): 0.00%


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

distilbert-base-uncased: longest 66 tokens, 0.00% over 128


... (2 outputs left out)

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

roberta-base: longest 66 tokens, 0.00% over 128


In [13]:
import json, os
os.makedirs(DATA, exist_ok=True)

COLS = ['id', 'text', 'label', 'source']
for split in ['train', 'val', 'test']:
    path = f'{DATA}/sst5_{split}.csv'
    clean[split][COLS].to_csv(path, index=False)
    check = pd.read_csv(path)
    assert check.shape == (len(clean[split]), 4) and check['text'].notna().all()
    print(f'Wrote {path}  ({len(check)} rows)')

counts = clean['train']['label'].value_counts().sort_index()
weights = {int(k): round(len(clean['train']) / (3 * v), 4) for k, v in counts.items()}
with open(f'{DATA}/class_weights.json', 'w') as f:
    json.dump({'source': 'sst5_train.csv', 'method': 'balanced', 'weights': weights}, f, indent=2)
print('Class weights:', weights)

Wrote /content/drive/MyDrive/capstone-ml/data/sst5_train.csv  (8544 rows)
Wrote /content/drive/MyDrive/capstone-ml/data/sst5_val.csv  (1101 rows)
Wrote /content/drive/MyDrive/capstone-ml/data/sst5_test.csv  (2210 rows)
Class weights: {0: 0.8604, 1: 1.7537, 2: 0.7889}


In [14]:
from datetime import date
from sklearn.metrics import accuracy_score, f1_score

MAJORITY = int(clean['train']['label'].value_counts().idxmax())
print('Majority training class:', MAJORITY, NAMES[MAJORITY])
os.makedirs(PRED, exist_ok=True)

def baseline_predictions(df, eval_set):
    p = [1.0 if c == MAJORITY else 0.0 for c in range(3)]
    return pd.DataFrame({
        'id': df['id'], 'true_label': df['label'], 'pred_label': MAJORITY,
        'p_neg': p[0], 'p_neu': p[1], 'p_pos': p[2],
        'model': 'majority', 'seed': '', 'train_data': 'sst5', 'eval_set': eval_set,
    })

results = {}
for split in ['val', 'test']:
    preds = baseline_predictions(clean[split], f'sst5_{split}')
    preds.to_csv(f'{PRED}/majority_sst5_{split}.csv', index=False)
    acc = accuracy_score(preds['true_label'], preds['pred_label'])
    mf1 = f1_score(preds['true_label'], preds['pred_label'], average='macro', zero_division=0)
    results[split] = mf1
    print(f'SST-5 {split}: accuracy {acc:.4f}, macro-F1 {mf1:.4f}')

log = f'{LOGS}/experiments.csv'
run_id = 'majority-sst5'
logged = pd.read_csv(log)
if run_id not in logged['run_id'].astype(str).values:
    pd.DataFrame([{'run_id': run_id, 'model': 'majority', 'seed': '', 'train_data': 'sst5',
                   'hyperparameters': f'always predicts {NAMES[MAJORITY]}',
                   'val_macro_f1': round(results['val'], 4), 'test_macro_f1': round(results['test'], 4),
                   'date': date.today().isoformat()}]).to_csv(log, mode='a', header=False, index=False)
    print('Logged', run_id)
else:
    print(run_id, 'already in the experiment log, not added again')

Majority training class: 2 positive
SST-5 val: accuracy 0.4033, macro-F1 0.1916
SST-5 test: accuracy 0.4113, macro-F1 0.1943
Logged majority-sst5


In [15]:
MARKET_IN = f'{DATA}/market_labelled.csv'
TO_ID = {'negative': 0, 'neutral': 1, 'positive': 2}

if not os.path.exists(MARKET_IN):
    print('market_labelled.csv not found. Skipping; rerun this notebook once the set is labelled.')
else:
    m = pd.read_csv(MARKET_IN)
    need = {'id', 'text', 'stars', 'direction', 'category', 'label'}
    assert need <= set(m.columns), f'missing columns: {need - set(m.columns)}'

    m['label'] = m['label'].str.strip().str.lower()
    removed = (m['label'] == 'remove').sum()
    m = m[m['label'] != 'remove'].copy()
    bad = set(m['label']) - set(TO_ID)
    assert not bad, f'unknown labels: {bad}'
    assert m['id'].is_unique, 'duplicate ids'

    m['text'] = m['text'].map(clean_text)
    m['label'] = m['label'].map(TO_ID)
    m['source'] = 'market'
    m = m[COLS + ['stars', 'direction', 'category']]

    counts = m['label'].map(NAMES).value_counts()
    print(f'{len(m)} reviews kept, {removed} removed, {m["text"].duplicated().sum()} duplicate texts')
    print(counts.to_string())
    if counts.min() < 100 or len(m) < 300:
        print('WARNING: below the spec of 300 reviews with at least 100 per class')

    mw = m['text'].str.split().str.len()
    print(f'Words per review: median {int(mw.median())}, longest {int(mw.max())}, over 64: {(mw > 64).mean():.1%}')

    m.to_csv(f'{DATA}/market_test.csv', index=False)
    preds = baseline_predictions(m, 'market_test')
    preds.to_csv(f'{PRED}/majority_market_test.csv', index=False)
    print('Marketplace baseline: accuracy', round(accuracy_score(preds['true_label'], preds['pred_label']), 4),
          '| macro-F1', round(f1_score(preds['true_label'], preds['pred_label'], average='macro', zero_division=0), 4))

market_labelled.csv not found. Skipping; rerun this notebook once the set is labelled.


In [16]:
import shutil, glob
from google.colab import files

bundle = '/content/ml_data'
shutil.rmtree(bundle, ignore_errors=True)
os.makedirs(bundle)
for f in glob.glob(f'{DATA}/sst5_*.csv') + glob.glob(f'{DATA}/market_test.csv') + [f'{DATA}/class_weights.json']:
    shutil.copy(f, bundle)
print(sorted(os.listdir(bundle)))
files.download(shutil.make_archive('/content/ml_data', 'zip', bundle))

['class_weights.json', 'sst5_test.csv', 'sst5_train.csv', 'sst5_val.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>